In [1]:
import time
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as Data
from torch.utils.data import TensorDataset # 텐서데이터셋
from torch.utils.data import DataLoader # 데이터로더
from torch.utils.data import Dataset
from sklearn.metrics import *
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from collections import defaultdict
import math

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
data_path = './aiffel/autoint/ml-1m'

In [4]:
movielens_rcmm = pd.read_csv(f"{data_path}/movielens_rcmm_v2.csv", dtype=str)
print(movielens_rcmm.shape)
movielens_rcmm.head()

(1000209, 15)


,user_id,movie_id,movie_decade,movie_year,rating_year,rating_month,rating_decade,genre1,genre2,genre3,gender,age,occupation,zip,label
0,1,1193,1970s,1975,2000,12,2000s,Drama,no,no,F,1,10,48067,1
1,1,661,1990s,1996,2000,12,2000s,Animation,Children's,Musical,F,1,10,48067,0
2,1,914,1960s,1964,2000,12,2000s,Musical,Romance,no,F,1,10,48067,0
3,1,3408,2000s,2000,2000,12,2000s,Drama,no,no,F,1,10,48067,1
4,1,2355,1990s,1998,2001,1,2000s,Animation,Children's,Comedy,F,1,10,48067,1


In [5]:
movielens_rcmm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 15 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   user_id        1000209 non-null  object
 1   movie_id       1000209 non-null  object
 2   movie_decade   1000209 non-null  object
 3   movie_year     1000209 non-null  object
 4   rating_year    1000209 non-null  object
 5   rating_month   1000209 non-null  object
 6   rating_decade  1000209 non-null  object
 7   genre1         1000209 non-null  object
 8   genre2         1000209 non-null  object
 9   genre3         1000209 non-null  object
 10  gender         1000209 non-null  object
 11  age            1000209 non-null  object
 12  occupation     1000209 non-null  object
 13  zip            1000209 non-null  object
 14  label          1000209 non-null  object
dtypes: object(15)
memory usage: 114.5+ MB


In [6]:
movielens_rcmm.isna().sum()

user_id          0
movie_id         0
movie_decade     0
movie_year       0
rating_year      0
rating_month     0
rating_decade    0
genre1           0
genre2           0
genre3           0
gender           0
age              0
occupation       0
zip              0
label            0
dtype: int64

In [7]:
label_encoders = {col: LabelEncoder() for col in movielens_rcmm.columns[:-1]} # label은 제외

# Apply Label Encoding
for col, le in label_encoders.items():
    movielens_rcmm[col] = le.fit_transform(movielens_rcmm[col])

In [8]:
for col, le in label_encoders.items():
    print(col, len(list(le.classes_)), list(le.classes_)[:10])

user_id 6040 ['1', '10', '100', '1000', '1001', '1002', '1003', '1004', '1005', '1006']
movie_id 3706 ['1', '10', '100', '1000', '1002', '1003', '1004', '1005', '1006', '1007']
movie_decade 10 ['1910s', '1920s', '1930s', '1940s', '1950s', '1960s', '1970s', '1980s', '1990s', '2000s']
movie_year 81 ['1919', '1920', '1921', '1922', '1923', '1925', '1926', '1927', '1928', '1929']
rating_year 4 ['2000', '2001', '2002', '2003']
rating_month 12 ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7']
rating_decade 1 ['2000s']
genre1 18 ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir']
genre2 18 ['Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror']
genre3 16 ['Animation', "Children's", 'Comedy', 'Crime', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery']
gender 2 ['F', 'M']
age 7 ['1', '18', '25', '35', '45', '50', '56']
occupation 21 ['0', '1', '10', '

In [9]:
movielens_rcmm.head()

,user_id,movie_id,movie_decade,movie_year,rating_year,rating_month,rating_decade,genre1,genre2,genre3,gender,age,occupation,zip,label
0,0,189,6,55,0,3,0,7,17,15,0,0,2,1588,1
1,0,3374,8,76,0,3,0,2,2,8,0,0,2,1588,0
2,0,3615,5,44,0,3,0,11,12,15,0,0,2,1588,0
3,0,2503,9,80,0,3,0,7,17,15,0,0,2,1588,1
4,0,1374,8,78,1,0,0,2,2,2,0,0,2,1588,1


In [10]:
train_df, test_df = train_test_split(movielens_rcmm, test_size=0.2, random_state=42)

In [11]:
print(train_df.shape)
print(test_df.shape)


(800167, 15)
(200042, 15)


In [12]:
u_i_feature = ['user_id', 'movie_id']
meta_features = ['movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1','genre2', 'genre3', 'gender', 'age', 'occupation', 'zip']
label = 'label'

In [13]:
class MvLensDataset(Dataset):
    def __init__(self, data, u_i_cols, label_col):
        self.n = data.shape[0]
        self.y = data[label_col].astype(np.float32).values.reshape(-1, 1)

        self.u_i_cols = u_i_cols
        self.feature_cols = [col for col in data.columns if col not in u_i_cols + [label_col]]
        
        self.data_v = data[self.u_i_cols + self.feature_cols].astype(np.int64).values

        self.field_dims = np.max(self.data_v, axis=0) + 1


    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return [self.data_v[idx], self.y[idx]]

In [14]:
%%time
train_dataset = MvLensDataset(data=train_df, u_i_cols=u_i_feature, label_col=label)
test_dataset = MvLensDataset(data=test_df, u_i_cols=u_i_feature, label_col=label)

CPU times: user 165 ms, sys: 68.2 ms, total: 233 ms
Wall time: 233 ms


In [15]:
print(train_dataset.u_i_cols)
print(train_dataset.feature_cols)
print(train_dataset.field_dims)

['user_id', 'movie_id']
['movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1', 'genre2', 'genre3', 'gender', 'age', 'occupation', 'zip']
[6040 3706   10   81    4   12    1   18   18   16    2    7   21 3439]


In [16]:
feature, l = train_dataset[:2]
print(feature, type(feature))
print(l)

[[5041 2076    8   79    0    8    0   15   17   15    1    2    8 2219]
 [4241  989    6   53    1    4    0   10   17   15    0    4   17  232]] <class 'numpy.ndarray'>
[[0.]
 [0.]]


In [17]:
class FeaturesEmbedding(torch.nn.Module):

    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.embedding = torch.nn.Embedding(sum(field_dims), embed_dim)
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.longlong)
        torch.nn.init.xavier_uniform_(self.embedding.weight.data)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)
        return self.embedding(x)

In [18]:
class MultiLayerPerceptron(nn.Module):
    def __init__(self, inputs_dim, hidden_units, activation='relu', l2_reg=0, dropout_rate=0, use_bn=False,
                 init_std=0.0001, dice_dim=3, output_layer=True, device='cpu'):
        super(MultiLayerPerceptron, self).__init__()
        self.dropout_rate = dropout_rate
        self.dropout = nn.Dropout(dropout_rate)
        self.l2_reg = l2_reg
        self.use_bn = use_bn
        hidden_units = [inputs_dim] + list(hidden_units)
        if output_layer:
            hidden_units += [1]
        activation_layer = nn.ReLU()
        

        self.linears = nn.ModuleList([nn.Linear(hidden_units[i], hidden_units[i + 1]) for i in range(len(hidden_units) - 1)])

        if self.use_bn:
            self.bn = nn.ModuleList([nn.BatchNorm1d(hidden_units[i + 1]) for i in range(len(hidden_units) - 1)])

        self.activation_layers = nn.ModuleList([activation_layer for i in range(len(hidden_units) - 1)])

        for name, tensor in self.linears.named_parameters():
            if 'weight' in name:
                nn.init.normal_(tensor, mean=0, std=init_std)

        self.to(device)

    def forward(self, inputs):
        model_input = inputs

        for i in range(len(self.linears)):

            fc = self.linears[i](model_input)

            if self.use_bn:
                fc = self.bn[i](fc)

            fc = self.activation_layers[i](fc)

            fc = self.dropout(fc)
            model_input = fc
        return model_input

In [19]:
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, embedding_size, head_num=2, use_res=True, scaling=False, device='cpu'):
        super(MultiHeadSelfAttention, self).__init__()
        if head_num <= 0:
            raise ValueError('head_num must be a int > 0')
        if embedding_size % head_num != 0:
            raise ValueError('embedding_size is not an integer multiple of head_num!')
        self.att_embedding_size = embedding_size // head_num
        self.head_num = head_num
        self.use_res = use_res
        self.scaling = scaling

        self.W_Query = nn.Parameter(torch.Tensor(embedding_size, embedding_size))
        self.W_key = nn.Parameter(torch.Tensor(embedding_size, embedding_size))
        self.W_Value = nn.Parameter(torch.Tensor(embedding_size, embedding_size))

        if self.use_res:
            self.W_Res = nn.Parameter(torch.Tensor(embedding_size, embedding_size))
        for tensor in self.parameters():
            nn.init.normal_(tensor, mean=0.0, std=0.05)

        self.to(device)

    def forward(self, inputs):

        if len(inputs.shape) != 3:
            raise ValueError(
                "Unexpected inputs dimensions %d, expect to be 3 dimensions" % (len(inputs.shape)))

        querys = torch.tensordot(inputs, self.W_Query, dims=([-1], [0]))
        keys = torch.tensordot(inputs, self.W_key, dims=([-1], [0]))
        values = torch.tensordot(inputs, self.W_Value, dims=([-1], [0]))

        # head_num None F D/head_num
        querys = torch.stack(torch.split(querys, self.att_embedding_size, dim=2))
        keys = torch.stack(torch.split(keys, self.att_embedding_size, dim=2))
        values = torch.stack(torch.split(values, self.att_embedding_size, dim=2))

        inner_product = torch.einsum('bnik,bnjk->bnij', querys, keys)
        if self.scaling:
            inner_product /= self.att_embedding_size ** 0.5
        self.normalized_att_scores = F.softmax(inner_product, dim=-1)
        result = torch.matmul(self.normalized_att_scores, values)

        result = torch.cat(torch.split(result, 1, ), dim=-1)
        result = torch.squeeze(result, dim=0)  # None F D
        if self.use_res:
            result += torch.tensordot(inputs, self.W_Res, dims=([-1], [0]))
        result = F.relu(result)

        return result

In [20]:
def test_model(model, test_loader):
    model.eval()
    user_pred_info = defaultdict(list)
    with torch.no_grad():
        with tqdm(test_loader, unit='batch') as tepoch:
            for samples in tepoch:
                features, y = samples[0], samples[1]
                features, y = features.to(device), y.to(device)
                y_pred = model(features)
                for feature, p in zip(features, y_pred):
                    u_i = feature[:2]
                    user_pred_info[int(u_i[0])].append((int(u_i[1]), float(p)))
    return user_pred_info

In [21]:
#https://github.com/xiangwang1223/neural_graph_collaborative_filtering/blob/master/NGCF/utility/metrics.py
def precision_at_k(r, k):
    assert k >= 1
    r = np.asarray(r)[:k]
    return np.mean(r)

def get_average_precision(r, yt_len, top=10):
    r = np.asarray(r)
    if r.size < top:
        top = r.size
    out = [precision_at_k(r, k + 1) for k in range(top) if r[k]]
    if not out:
        return 0.
    return round( np.sum(out)/float(min(top, yt_len)), 5)


def mean_average_precision(rs, top):
    return np.mean([get_average_precision(r, top) for r in rs])

#https://www.programcreek.com/python/?code=MaurizioFD%2FRecSys2019_DeepLearning_Evaluation%2FRecSys2019_DeepLearning_Evaluation-master%2FConferences%2FKDD%2FMCRec_our_interface%2FMCRecRecommenderWrapper.py
def get_DCG(ranklist, y_true):
    dcg = 0.0
    for i in range(len(ranklist)):
        item = ranklist[i]
        if item in y_true:
            dcg += 1.0 / math.log(i + 2)
    return  dcg

def get_IDCG(ranklist, y_true):
    idcg = 0.0
    i = 0
    for item in y_true:
        if item in ranklist:
            idcg += 1.0 / math.log(i + 2)
            i += 1
    return idcg

def get_NDCG(ranklist, y_true):
    ranklist = np.array(ranklist).astype(int)
    y_true = np.array(y_true).astype(int)
    dcg = get_DCG(ranklist, y_true)
    idcg = get_IDCG(y_true, y_true)
    if idcg == 0:
        return 0
    return round( (dcg / idcg), 5)

def get_hit_rate(ranklist, y_true):
    c = 0
    for y in y_true:
        if y in ranklist:
            c += 1
    return round( c / len(y_true), 5 )

In [22]:
field_dims = train_dataset.field_dims
epoch= 5
learning_rate= 0.0001
dropout= 0.4
batch_size = 512
embed_dim= 16

In [28]:
class AutoIntMLP(nn.Module):

    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=0.4, init_std=0.0001, device='cpu'):

        super(AutoIntMLP, self).__init__()
        self.embedding = FeaturesEmbedding(field_dims, embed_dim)
        self.num_fields = len(field_dims)
        self.embedding_size = embedding_size
        self.att_output_dim = self.num_fields * self.embedding_size
        self.embed_output_dim = len(field_dims) * embed_dim

        self.dnn_linear = nn.Linear(self.att_output_dim, 1, bias=False).to(device)
        self.dnn_hidden_units = dnn_hidden_units
        self.att_layer_num = att_layer_num
        self.dnn = MultiLayerPerceptron(self.embed_output_dim, dnn_hidden_units,
                           activation=dnn_activation, l2_reg=l2_reg_dnn, dropout_rate=dnn_dropout, use_bn=dnn_use_bn,
                           init_std=init_std, output_layer=True, device=device)
        self.int_layers = nn.ModuleList(
            [MultiHeadSelfAttention(self.embedding_size, att_head_num, att_res, device=device) for _ in range(att_layer_num)])

        self.to(device)

    def forward(self, X):
        embed_x = self.embedding(X)
        dnn_embed = embed_x
        att_input = embed_x
        
        for cnt, layer in enumerate(self.int_layers):
            att_input = layer(att_input)
            
        att_output = torch.flatten(att_input, start_dim=1)
        
        att_output = F.relu(self.dnn_linear(att_output))
        # autoint MLP
        dnn_output = self.dnn(dnn_embed.view(-1, self.embed_output_dim))
        
        y_pred = torch.sigmoid(att_output + dnn_output)

        return y_pred

In [29]:
field_dims = train_dataset.field_dims
epoch= 5
learning_rate= 0.0001
dropout= 0.4
batch_size = 512
embed_dim= 16

In [30]:
model = AutoIntMLP(field_dims, embed_dim, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=dropout, init_std=0.0001, device=device).to(device)

In [31]:
train_mymodel_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_mymodel_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
loss_fn = nn.BCELoss()
mymodel_optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [32]:
for epoch in range(epoch):
    model.train()
    epoch_loss = 0
    epoch_accuracy = 0
    with tqdm(train_mymodel_dataloader, unit='batch') as tepoch:
        for samples in tepoch:
            tepoch.set_description(f"Epoch {epoch}")
            mymodel_optimizer.zero_grad()
            features, y = samples[0], samples[1]
            features, y = features.to(device), y.to(device)
            y_pred = model(features)
            loss = loss_fn(y_pred.reshape(-1, 1), y)
            epoch_loss += loss
            loss.backward()
            mymodel_optimizer.step() 
            tepoch.set_postfix(loss=loss.item())


Epoch 4: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1563/1563 [00:37<00:00, 41.95batch/s, loss=0.603]


In [33]:
user_pred_info = {}
top = 10

mymodel_user_pred_info = test_model(model, test_mymodel_dataloader)

for user, data_info in tqdm(mymodel_user_pred_info.items(), total=len(mymodel_user_pred_info), position=0, leave=True):
    ranklist = sorted(data_info, key=lambda s : s[1], reverse=True)[:top]
    ranklist = list(dict.fromkeys([r[0] for r in ranklist]))
    user_pred_info[str(user)] = ranklist
test_data = test_df[test_df['label']=='1'].groupby('user_id')['movie_id'].apply(list)


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6035/6035 [00:00<00:00, 90407.72it/s]


In [34]:
mymodel_ndcg_result = {}
mymodel_hitrate_result = {}

for user, data_info in tqdm(test_data.items(), total=len(test_data), position=0, leave=True):
    mymodel_pred = user_pred_info.get(str(user))

    testset = list(set(np.array(data_info).astype(int)))
    mymodel_pred = mymodel_pred[:top]

    # NDCG 값 구하기
    user_ndcg = get_NDCG(mymodel_pred, testset)

    mymodel_ndcg_result[user] = user_ndcg
    
for user, data_info in tqdm(test_data.items(), total=len(test_data), position=0, leave=True):
    # model pred 값과 test 값을 가져오되, 타입을 맞춰 줌
    mymodel_pred = user_pred_info.get(str(user))

    testset = list(set(np.array(data_info).astype(int)))
    mymodel_pred = mymodel_pred[:top]

    # hitrate 값 구하기
    user_hitrate = get_hit_rate(mymodel_pred, testset)

    # 사용자 hitrate 결과 저장
    mymodel_hitrate_result[user] = user_hitrate


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5994/5994 [00:00<00:00, 68770.93it/s]


In [35]:

print(" mymodel ndcg : ", round(np.mean(list(mymodel_ndcg_result.values())), 5))
print(" mymodel hitrate : ", round(np.mean(list(mymodel_hitrate_result.values())), 5))

 mymodel ndcg :  0.66011
 mymodel hitrate :  0.63024
